In [7]:
import json
import floorplan_backend
import time

In [8]:
with open('./samples/debug_design.json') as topology_json:
    with open('./samples/construction.json') as construction_json:
        topology = json.load(topology_json)
        construction = json.load(construction_json)
        start = time.time()
        building = floorplan_backend.Building.from_data(topology, construction)
        print(time.time()-start)
        building.solve_joints()
        geoms = [building.layer_geometry(face_id) for face_id in building.face_ids()]

0.0010173320770263672


In [9]:
# Orientation diagnostic for the foundation interfaces.
# The JSON fixture contains the post-canonicalization uvn, so it cannot recover
# the original Topologic normal. It can still show stored normal, loop winding,
# and every incident face's direction along each foundation edge.
import numpy as np

verts = topology["verts"]
faces = topology["faces"]
edges = topology["edges"]


def direction_on_face(face, edge):
    a, b = edge["verts"]
    loop = face["verts"]
    for i, v0 in enumerate(loop):
        v1 = loop[(i + 1) % len(loop)]
        if (v0, v1) == (a, b):
            return "same"
        if (v0, v1) == (b, a):
            return "opposite"
    return "edge missing from face loop"


for face_id, face in faces.items():
    if face["type"].lower() != "foundation":
        continue
    pts = np.array([verts[v]["coords"] for v in face["verts"]], dtype=float)
    area_vec = sum(np.cross(pts[i], pts[(i + 1) % len(pts)]) for i in range(len(pts)))
    stored_n = np.array(face["uvn"][2], dtype=float)
    stored_n /= np.linalg.norm(stored_n)
    print(f"\nFoundation {face_id}: loop·stored_normal={np.dot(area_vec, stored_n):.6g}, stored_normal={stored_n}")
    for edge_id in face["edges"]:
        edge = edges[edge_id]
        own_dir = direction_on_face(face, edge)
        neighbors = []
        for other_id in edge["faces"]:
            if other_id == face_id:
                continue
            other = faces[other_id]
            neighbors.append(f"{other['type']} {other_id}: {direction_on_face(other, edge)}")
        print(f"  edge {edge_id} (incident={len(edge['faces'])}): foundation={own_dir}; " + "; ".join(neighbors))



Foundation 3dc14b0f-b60f-4299-a36a-0fcf95bec30e: loop·stored_normal=54.3367, stored_normal=[ 0. -0.  1.]
  edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816 (incident=3): foundation=opposite; Interior wall eef2fc3e-621a-4536-ac3a-3d3af0018637: opposite; Foundation cff21a44-de05-43ab-8f90-5535a2fd0892: same
  edge 7617b987-1358-4918-8420-f23021ddcacd (incident=2): foundation=opposite; Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca: opposite
  edge e3850b1e-6169-4d49-af15-3b0259a592bd (incident=2): foundation=opposite; Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca: opposite
  edge 33f2fbf4-fb88-4ec6-839e-b5229f40abd1 (incident=2): foundation=opposite; Facade 07190a37-a44c-4aec-96b4-0b5d753ea965: opposite
  edge eaaa4488-e5c3-4479-bf40-9ed77bebd32a (incident=2): foundation=opposite; Facade 07190a37-a44c-4aec-96b4-0b5d753ea965: opposite
  edge a5174a5d-06fc-4e59-8948-dca6f1c20a7a (incident=2): foundation=opposite; Facade 1a91cf73-8621-4832-ab54-f0e2477716c7: opposite
  edge 12ead49f-c70d-4b7e-8021-5be

In [10]:
# Inspect the actual joint-material assignment on foundation edge/layer 0.
# Face IDs are internal arena IDs, so associate each one to the closest
# foundation polygon by centroid in XY before comparing to the JSON topology.
import numpy as np

foundation_faces = {
    fid: face for fid, face in topology["faces"].items()
    if face["type"].lower() == "foundation"
}
face_centers = {
    fid: np.mean([topology["verts"][v]["coords"] for v in face["verts"]], axis=0)
    for fid, face in foundation_faces.items()
}

for py_id, geom in zip(building.face_ids(), geoms):
    if not geom:
        continue
    layer0 = geom[0]
    layer0_center = np.mean(layer0["inner"], axis=0)
    nearest_id = min(
        face_centers,
        key=lambda fid: np.linalg.norm(layer0_center[:2] - face_centers[fid][:2]),
    )
    if np.linalg.norm(layer0_center[:2] - face_centers[nearest_id][:2]) > 1.0:
        continue

    face = foundation_faces[nearest_id]
    edge_materials = layer0["edge_materials"]
    print(f"\nFoundation {nearest_id} / arena id ({py_id.index}, {py_id.generation})")
    print("layer0:", layer0["name"], layer0["uuid"])
    for edge_idx, edge_id in enumerate(face["edges"]):
        edge = topology["edges"][edge_id]
        assigned = edge_materials[0][edge_idx]
        incident = [faces[fid]["type"] + " " + fid for fid in edge["faces"] if fid != nearest_id]
        print(f"  edge {edge_id} incident={len(edge['faces'])} layer0_joint_material={assigned}; {incident}")



Foundation 3dc14b0f-b60f-4299-a36a-0fcf95bec30e / arena id (1, 0)
layer0: roof_insulation 81103e84-4d8d-4db5-88a5-ab03659ec0d5
  edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816 incident=3 layer0_joint_material=None; ['Interior wall eef2fc3e-621a-4536-ac3a-3d3af0018637', 'Foundation cff21a44-de05-43ab-8f90-5535a2fd0892']
  edge 7617b987-1358-4918-8420-f23021ddcacd incident=2 layer0_joint_material=db573df3-b731-493d-9e3d-166cacd7424e; ['Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca']
  edge e3850b1e-6169-4d49-af15-3b0259a592bd incident=2 layer0_joint_material=db573df3-b731-493d-9e3d-166cacd7424e; ['Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca']
  edge 33f2fbf4-fb88-4ec6-839e-b5229f40abd1 incident=2 layer0_joint_material=db573df3-b731-493d-9e3d-166cacd7424e; ['Facade 07190a37-a44c-4aec-96b4-0b5d753ea965']
  edge eaaa4488-e5c3-4479-bf40-9ed77bebd32a incident=2 layer0_joint_material=db573df3-b731-493d-9e3d-166cacd7424e; ['Facade 07190a37-a44c-4aec-96b4-0b5d753ea965']
  edge a5174a5d-06fc-4e59-8948-

In [11]:
# Reproduce the solver's layer-scan decision for each foundation layer 0.
# This makes the candidate order and the first priority-triggered pairing visible.
layer_groups = construction["layer_groups"]
layers = construction["layers"]


def layer_group_for(face):
    group_id = face["construction"]
    return layer_groups[group_id]


for face_id, face in topology["faces"].items():
    if face["type"].lower() != "foundation":
        continue
    foundation_group = layer_group_for(face)
    foundation_layer0 = layers[foundation_group["layers"][0]]
    priority0 = foundation_layer0["priority"]
    scan_cw = any(
        layers[layer_id]["priority"] > priority0
        for layer_id in foundation_group["layers"][1:]
    )
    print(f"\nFoundation {face_id}: layer0={foundation_layer0['name']} priority={priority0}, scan_cw={scan_cw}")

    for edge_id in face["edges"]:
        edge = edges[edge_id]
        other_ids = [other_id for other_id in edge["faces"] if other_id != face_id]
        for other_id in other_ids:
            other = faces[other_id]
            if other["type"].lower() == "foundation":
                continue
            foundation_is_ccw = direction_on_face(face, edge) == "same"
            other_is_ccw = direction_on_face(other, edge) == "same"
            flip_layers = scan_cw ^ (foundation_is_ccw != other_is_ccw)
            candidate_group = layer_group_for(other)
            candidate_indices = list(range(len(candidate_group["layers"])))
            if flip_layers:
                candidate_indices.reverse()
            candidate_scan = [
                (i, layers[candidate_group["layers"][i]]["name"],
                 layers[candidate_group["layers"][i]]["priority"])
                for i in candidate_indices
            ]
            solver_choice = None
            for candidate in candidate_scan:
                if candidate[2] > priority0:
                    solver_choice = (candidate, "higher-priority butt")
                    break
                if int(candidate[2]) == int(priority0):
                    solver_choice = (candidate, "same-priority-class joint")
                    break
            print(f"  edge {edge_id} -> {other['type']} {other_id}: flip_layers={flip_layers}, scan={candidate_scan}, solver_choice={solver_choice}")



Foundation 3dc14b0f-b60f-4299-a36a-0fcf95bec30e: layer0=roof_insulation priority=80.1, scan_cw=True
  edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816 -> Interior wall eef2fc3e-621a-4536-ac3a-3d3af0018637: flip_layers=True, scan=[(2, 'plate', 90), (1, 'studs', 100), (0, 'plate', 90)], solver_choice=((2, 'plate', 90), 'higher-priority butt')
  edge 7617b987-1358-4918-8420-f23021ddcacd -> Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca: flip_layers=True, scan=[(4, 'facade', 20), (3, 'wall_insulation', 80), (2, 'plate', 90), (1, 'studs', 100), (0, 'plate', 90)], solver_choice=((3, 'wall_insulation', 80), 'same-priority-class joint')
  edge e3850b1e-6169-4d49-af15-3b0259a592bd -> Facade cb0c6ec8-b153-4458-b0e3-9c649d7021ca: flip_layers=True, scan=[(4, 'facade', 20), (3, 'wall_insulation', 80), (2, 'plate', 90), (1, 'studs', 100), (0, 'plate', 90)], solver_choice=((3, 'wall_insulation', 80), 'same-priority-class joint')
  edge 33f2fbf4-fb88-4ec6-839e-b5229f40abd1 -> Facade 07190a37-a44c-4aec-96b4-0b5

In [12]:
geoms = [building.layer_geometry(face_id) for face_id in building.face_ids()]
for geom in geoms:
    if geom[1]['name'] == 'studs':
        print('all of these zeros should be .2 the rest unchanged',geom[0]['inner'][:,2])
        #all of these zeros should be .2 the rest unchanged
    else:
        for layer in geom:
            print('remain unchanged',layer['inner'],layer['outer'])
            #all of these should remain unchanged


all of these zeros should be .2 the rest unchanged [0.         0.         1.63954071 3.27908142 3.27908142 3.27908142
 1.63954071 0.        ]
remain unchanged [[-3.24360970e+02  8.08826032e+01 -1.00000000e-01]
 [-3.26604022e+02  8.08826032e+01 -1.00000000e-01]
 [-3.29022356e+02  8.08826032e+01 -1.00000000e-01]
 [-3.29114017e+02  7.71529742e+01 -1.00000000e-01]
 [-3.29205943e+02  7.34125349e+01 -1.00000000e-01]
 [-3.27319554e+02  7.34125349e+01 -1.00000000e-01]
 [-3.25698016e+02  7.34125349e+01 -1.00000000e-01]
 [-3.25029493e+02  7.71475690e+01 -1.00000000e-01]] [[-324.36096951   80.88260317    0.        ]
 [-326.60402151   80.88260317    0.        ]
 [-329.02235598   80.88260317    0.        ]
 [-329.11401661   77.15297421    0.        ]
 [-329.20594291   73.41253492    0.        ]
 [-327.31955408   73.41253492    0.        ]
 [-325.69801566   73.41253492    0.        ]
 [-325.02949258   77.14756905    0.        ]]
remain unchanged [[-324.26890998   80.77260317    0.        ]
 [-326.60

In [13]:
# Check the wall's layer 0 assignment on edges shared with a foundation.
# Match geometry results to source faces by layer-group signature and 3D centroid.
import numpy as np

face_groups = {}
for face_id, face in faces.items():
    group = construction["layer_groups"][face["construction"]]
    face_groups[face_id] = tuple(group["layers"])

geometry_records = []
for py_id, geom in zip(building.face_ids(), geoms):
    signature = tuple(layer["uuid"] for layer in geom)
    center = np.mean(geom[0]["inner"], axis=0)
    geometry_records.append((py_id, geom, signature, center))

for edge_id, edge in edges.items():
    foundation_ids = [fid for fid in edge["faces"] if faces[fid]["type"].lower() == "foundation"]
    wall_ids = [fid for fid in edge["faces"] if faces[fid]["type"].lower() in ("facade", "interior wall")]
    if not foundation_ids or not wall_ids:
        continue
    for wall_id in wall_ids:
        wall = faces[wall_id]
        signature = face_groups[wall_id]
        wall_center = np.mean([verts[v]["coords"] for v in wall["verts"]], axis=0)
        candidates = [r for r in geometry_records if r[2] == signature]
        if not candidates:
            print("No geometry match for wall", wall_id)
            continue
        py_id, geom, _, _ = min(candidates, key=lambda r: np.linalg.norm(r[3] - wall_center))
        edge_idx = wall["edges"].index(edge_id)
        assigned = geom[0]["edge_materials"][0][edge_idx]
        print(f"edge {edge_id}: wall {wall_id} layer0={geom[0]['name']} joint_material={assigned}; foundations={foundation_ids}; arena id=({py_id.index}, {py_id.generation})")


edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816: wall eef2fc3e-621a-4536-ac3a-3d3af0018637 layer0=plate joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e', 'cff21a44-de05-43ab-8f90-5535a2fd0892']; arena id=(7, 0)
edge fdac8729-9876-45b9-8ed0-6c6eb55416b1: wall eef2fc3e-621a-4536-ac3a-3d3af0018637 layer0=plate joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e', 'cff21a44-de05-43ab-8f90-5535a2fd0892']; arena id=(7, 0)
edge 12ead49f-c70d-4b7e-8021-5becf364ad6d: wall 1a91cf73-8621-4832-ab54-f0e2477716c7 layer0=plate joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e']; arena id=(5, 0)
edge a5174a5d-06fc-4e59-8948-dca6f1c20a7a: wall 1a91cf73-8621-4832-ab54-f0e2477716c7 layer0=plate joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e']; arena id=(5, 0)
edge eaaa4488-e5c3-4479-bf40-9ed77be

In [14]:
# Counterfactual: restore the known-original downward foundation normals and
# reverse their face loops back together. Compare wall layer 0 at slab interfaces.
import copy
import numpy as np


def make_runtime_geometries(input_topology):
    test_building = floorplan_backend.Building.from_data(input_topology, construction)
    test_building.solve_joints()
    test_geometries = [
        test_building.layer_geometry(face_id)
        for face_id in test_building.face_ids()
    ]
    return test_building, test_geometries


def geometry_for_source_face(source_face_id, source_faces, source_verts, runtime_building, runtime_geometries):
    source_face = source_faces[source_face_id]
    signature = tuple(construction["layer_groups"][source_face["construction"]]["layers"])
    center = np.mean([source_verts[v]["coords"] for v in source_face["verts"]], axis=0)
    candidates = []
    for py_id, geom in zip(runtime_building.face_ids(), runtime_geometries):
        if tuple(layer["uuid"] for layer in geom) == signature:
            geom_center = np.mean(geom[0]["inner"], axis=0)
            candidates.append((np.linalg.norm(geom_center - center), geom))
    if not candidates:
        raise RuntimeError(f"No runtime geometry matched source face {source_face_id}")
    distance, result = min(candidates, key=lambda item: item[0])
    if distance > 2.0:
        raise RuntimeError(f"Uncertain face match for {source_face_id}: centroid distance {distance}")
    return result


# Reconstruct the pre-canonicalized direction: downward normal and reversed loop.
topology_down = copy.deepcopy(topology)
for face in topology_down["faces"].values():
    if face["type"].lower() != "foundation":
        continue
    face["verts"] = [face["verts"][0]] + list(reversed(face["verts"][1:]))
    face["edges"] = list(reversed(face["edges"]))
    face["uvn"][2] = [-x for x in face["uvn"][2]]

original_building, original_geometries = building, geoms
down_building, down_geometries = make_runtime_geometries(topology_down)

for wall_id, wall in faces.items():
    if wall["type"].lower() not in ("facade", "interior wall"):
        continue
    for edge_id in wall["edges"]:
        edge = edges[edge_id]
        foundation_ids = [fid for fid in edge["faces"] if faces[fid]["type"].lower() == "foundation"]
        if not foundation_ids:
            continue
        edge_idx = wall["edges"].index(edge_id)
        geom_up = geometry_for_source_face(wall_id, faces, verts, original_building, original_geometries)
        geom_down = geometry_for_source_face(wall_id, faces, verts, down_building, down_geometries)
        mat_up = geom_up[0]["edge_materials"][0][edge_idx]
        mat_down = geom_down[0]["edge_materials"][0][edge_idx]
        corners_up = (geom_up[0]["inner"][edge_idx], geom_up[0]["outer"][edge_idx])
        corners_down = (geom_down[0]["inner"][edge_idx], geom_down[0]["outer"][edge_idx])
        print(f"wall {wall_id} layer0={geom_up[0]['name']} edge {edge_id} foundations={foundation_ids}")
        print(f"  canonical +Z: joint_material={mat_up}; inner/outer start={corners_up}")
        print(f"  restored -Z:  joint_material={mat_down}; inner/outer start={corners_down}")

# Also compare the foundation slab itself (layer index 1) in both variants.
for foundation_id, foundation in faces.items():
    if foundation["type"].lower() != "foundation":
        continue
    geom_up = geometry_for_source_face(foundation_id, faces, verts, original_building, original_geometries)
    geom_down = geometry_for_source_face(foundation_id, faces, verts, down_building, down_geometries)
    slab_up = np.concatenate([geom_up[1]["inner"], geom_up[1]["outer"]], axis=0)
    slab_down = np.concatenate([geom_down[1]["inner"], geom_down[1]["outer"]], axis=0)
    print(f"\nFoundation {foundation_id} slab {geom_up[1]['name']}:")
    print(f"  canonical normal: z=[{slab_up[:, 2].min():.6f}, {slab_up[:, 2].max():.6f}]")
    print(f"  restored normal:  z=[{slab_down[:, 2].min():.6f}, {slab_down[:, 2].max():.6f}]")


wall eef2fc3e-621a-4536-ac3a-3d3af0018637 layer0=plate edge fdac8729-9876-45b9-8ed0-6c6eb55416b1 foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e', 'cff21a44-de05-43ab-8f90-5535a2fd0892']
  canonical +Z: joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; inner/outer start=(array([-325.02949342,   77.1475692 ,    0.        ]), array([-325.01964986,   77.14580733,    0.        ]))
  restored -Z:  joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; inner/outer start=(array([-325.02949342,   77.1475692 ,    0.        ]), array([-325.01964986,   77.14580733,    0.        ]))
wall eef2fc3e-621a-4536-ac3a-3d3af0018637 layer0=plate edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816 foundations=['3dc14b0f-b60f-4299-a36a-0fcf95bec30e', 'cff21a44-de05-43ab-8f90-5535a2fd0892']
  canonical +Z: joint_material=b0fdd36a-18c3-4809-9cbf-c192eabfa156; inner/outer start=(array([-324.40034837,   80.66260347,    0.        ]), array([-324.38839958,   80.67260347,    0.        ]))
  restored -Z:  joint_material=b

In [15]:
# Trace wall layer 0 -> foundation slab selection and the exact Butt branch.
# This mirrors Building::solve_joint and Building::apply_joint without editing Rust.
import math
import numpy as np


def norm3(v):
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)
    return v / length if length > 1e-15 else np.zeros(3)


def cross3(a, b):
    return np.cross(a, b)


def work_frame(origin, normal):
    n = norm3(normal)
    if abs(n[0]) <= abs(n[1]) and abs(n[0]) <= abs(n[2]):
        u = np.array([0.0, -n[2], n[1]])
    elif abs(n[1]) <= abs(n[2]):
        u = np.array([-n[2], 0.0, n[0]])
    else:
        u = np.array([-n[1], n[0], 0.0])
    u = norm3(u)
    v = norm3(cross3(n, u))
    return np.asarray(origin, dtype=float), u, v


def project_point(frame, point):
    origin, u, v = frame
    d = np.asarray(point, dtype=float) - origin
    return np.array([np.dot(d, u), np.dot(d, v)])


def project_vector(frame, vector):
    _, u, v = frame
    return np.array([np.dot(vector, u), np.dot(vector, v)])


def line_intersection(a, da, b, db):
    den = da[0] * db[1] - da[1] * db[0]
    if abs(den) < 1e-12:
        return None
    q = b - a
    t = (q[0] * db[1] - q[1] * db[0]) / den
    return a + t * da


def trace_wall_plate_to_foundation(source_topology, wall_id, foundation_id, edge_id):
    source_faces = source_topology["faces"]
    source_edges = source_topology["edges"]
    source_verts = source_topology["verts"]
    wall = source_faces[wall_id]
    foundation = source_faces[foundation_id]
    edge = source_edges[edge_id]
    wall_group = construction["layer_groups"][wall["construction"]]
    foundation_group = construction["layer_groups"][foundation["construction"]]
    wall_layer_idx = 0
    foundation_layer_idx = 1
    wall_layer = construction["layers"][wall_group["layers"][wall_layer_idx]]
    slab_layer = construction["layers"][foundation_group["layers"][foundation_layer_idx]]
    wall_priority = wall_layer["priority"]
    scan_cw = any(
        construction["layers"][layer_id]["priority"] > wall_priority
        for layer_id in wall_group["layers"][wall_layer_idx + 1:]
    )
    edge_v0, edge_v1 = edge["verts"]
    wall_v0 = wall["verts"].index(edge_v0)
    foundation_v0 = foundation["verts"].index(edge_v0)
    wall_reversed = edge_v1 != wall["verts"][(wall_v0 + 1) % len(wall["verts"])]
    foundation_reversed = edge_v1 != foundation["verts"][(foundation_v0 + 1) % len(foundation["verts"])]
    wall_is_ccw = not wall_reversed
    foundation_is_ccw = not foundation_reversed
    flip_layers = scan_cw ^ (wall_is_ccw != foundation_is_ccw)
    candidate_indices = list(range(len(foundation_group["layers"])))
    if flip_layers:
        candidate_indices.reverse()

    chosen = None
    for candidate_idx in candidate_indices:
        candidate = construction["layers"][foundation_group["layers"][candidate_idx]]
        if candidate["priority"] > wall_priority:
            chosen = (candidate_idx, candidate, "higher-priority butt")
            break
        if int(candidate["priority"]) == int(wall_priority):
            chosen = (candidate_idx, candidate, "same-priority-class")
            break

    if chosen is None or chosen[0] != foundation_layer_idx:
        return f"edge {edge_id}: scan selected {None if chosen is None else (chosen[0], chosen[1]['name'])}, expected slab index {foundation_layer_idx}"

    p0 = np.array(source_verts[edge_v0]["coords"], dtype=float)
    p1 = np.array(source_verts[edge_v1]["coords"], dtype=float)
    edge_dir = norm3(p1 - p0)
    joint_frame = work_frame(p0, edge_dir)
    wall_normal = norm3(wall["uvn"][2])
    foundation_normal = norm3(foundation["uvn"][2])
    wall_edge_normal = cross3(edge_dir, wall_normal) * (-1.0 if wall_reversed else 1.0)
    foundation_edge_normal = cross3(edge_dir, foundation_normal) * (-1.0 if foundation_reversed else 1.0)
    n0 = norm3(project_vector(joint_frame, wall_edge_normal))
    n1 = norm3(project_vector(joint_frame, foundation_edge_normal))

    wall_offset = wall_group["layer_offsets"][0]
    wall_thickness = wall_layer["thickness"]
    slab_offset = foundation_group["layer_offsets"][foundation_layer_idx]
    slab_thickness = slab_layer["thickness"]
    p01 = project_point(joint_frame, p0 + wall_normal * wall_offset)
    p02 = project_point(joint_frame, p0 + wall_normal * (wall_offset + wall_thickness))
    p11 = project_point(joint_frame, p0 + foundation_normal * slab_offset)
    p12 = project_point(joint_frame, p0 + foundation_normal * (slab_offset + slab_thickness))
    i11 = line_intersection(p01, n0, p11, n1)
    i22 = line_intersection(p02, n0, p12, n1)
    i12 = line_intersection(p01, n0, p12, n1)
    i21 = line_intersection(p02, n0, p11, n1)
    if any(i is None for i in (i11, i22, i12, i21)):
        return f"edge {edge_id}: parallel lines prevented Butt intersections"

    dot = float(np.dot(n0, n1))
    cross = float(n0[0] * n1[1] - n0[1] * n1[0])
    flip = (math.atan2(cross, dot) % (2 * math.pi)) < math.pi
    if wall_reversed == foundation_reversed:
        inner_name, outer_name = "i11 (slab inner)", "i21 (slab inner)"
    else:
        rev0, rev1 = wall_reversed, foundation_reversed
        if flip:
            rev0, rev1 = not rev0, not rev1
        if rev0 and not rev1:
            inner_name, outer_name = "i22 (slab outer)", "i12 (slab outer)"
        else:
            inner_name, outer_name = "i11 (slab inner)", "i21 (slab inner)"
    return (f"edge {edge_id}: scan_cw={scan_cw}, flip_layers={flip_layers}, candidate_order={candidate_indices}, "
            f"selected slab={chosen[0]} ({chosen[1]['name']}), wall_rev={wall_reversed}, slab_rev={foundation_reversed}, "
            f"flip={flip}, Butt selects {inner_name}/{outer_name}, slab inner/outer z={p11[1]:.6f}/{p12[1]:.6f}")


for variant, source_topology in (("canonical", topology), ("restored", topology_down)):
    print(f"\n--- {variant} orientation ---")
    source_faces = source_topology["faces"]
    source_edges = source_topology["edges"]
    for wall_id, wall in source_faces.items():
        if wall["type"].lower() not in ("facade", "interior wall"):
            continue
        for edge_id in wall["edges"]:
            for foundation_id in source_edges[edge_id]["faces"]:
                if source_faces[foundation_id]["type"].lower() == "foundation":
                    print(trace_wall_plate_to_foundation(source_topology, wall_id, foundation_id, edge_id))



--- canonical orientation ---
edge fdac8729-9876-45b9-8ed0-6c6eb55416b1: scan_cw=True, flip_layers=True, candidate_order=[1, 0], selected slab=1 (foundation_slab), wall_rev=True, slab_rev=True, flip=True, Butt selects i11 (slab inner)/i21 (slab inner), slab inner/outer z=0.000000/0.200000
edge fdac8729-9876-45b9-8ed0-6c6eb55416b1: scan_cw=True, flip_layers=False, candidate_order=[0, 1], selected slab=1 (foundation_slab), wall_rev=True, slab_rev=False, flip=False, Butt selects i22 (slab outer)/i12 (slab outer), slab inner/outer z=0.000000/0.200000
edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816: scan_cw=True, flip_layers=True, candidate_order=[1, 0], selected slab=1 (foundation_slab), wall_rev=True, slab_rev=True, flip=True, Butt selects i11 (slab inner)/i21 (slab inner), slab inner/outer z=0.000000/0.200000
edge b421576d-92e2-43e1-a1ba-dcbd7a4fb816: scan_cw=True, flip_layers=False, candidate_order=[0, 1], selected slab=1 (foundation_slab), wall_rev=True, slab_rev=False, flip=False, Butt sel